# Local evaluator

Fast end-to-end smoke test for the Biohub UNet3D + transformer + ILP pipeline. It selects the shortest labeled training video, predicts its graph, and evaluates it with the metric implementation shipped in the public support artifact.

**Interpretation warning:** this run proves that inference and scoring work. It is not yet an unbiased validation score because the public checkpoint may have seen this video, and this first version evaluates the raw predicted graph before the leaderboard notebook's additional graph repair.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, zipfile

COMPETITION = 'biohub-cell-tracking-during-development'
TRAIN_DIR = Path(f'/kaggle/input/competitions/{COMPETITION}/train')
WORK_DIR = Path('/kaggle/working/biohub_local_eval')
REPO_DIR = WORK_DIR / 'repo'
DET_THRESHOLD = 0.97
UNET_BATCH_SIZE = 4
MAX_DISTANCE_UM = 7.0

assert TRAIN_DIR.exists(), f'Missing competition train mount: {TRAIN_DIR}'
print('TRAIN_DIR:', TRAIN_DIR)
print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

In [ ]:
# Locate the attached pilkwang support artifact.
manifest_candidates = list(Path('/kaggle/input').glob('**/ARTIFACT_MANIFEST.json'))
support_manifest = None
for candidate in manifest_candidates:
    try:
        meta = json.loads(candidate.read_text())
    except Exception:
        continue
    if meta.get('model', {}).get('method') == 'unet_transformer' and (candidate.parent / 'repo').exists():
        support_manifest = candidate
        break
assert support_manifest is not None, 'Attach pilkwang/biohub-tracking-support-pack-50ep-v1'
SUPPORT_DIR = support_manifest.parent
manifest = json.loads(support_manifest.read_text())
print('Support:', SUPPORT_DIR)
print('Artifact:', manifest.get('artifact_name'))
print('Weight sha256:', manifest.get('model', {}).get('weight_sha256'))

shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(parents=True)
if (SUPPORT_DIR / 'repo').is_dir():
    shutil.copytree(SUPPORT_DIR / 'repo', REPO_DIR)
else:
    REPO_DIR.mkdir()
    with zipfile.ZipFile(SUPPORT_DIR / 'repo.zip') as archive:
        archive.extractall(REPO_DIR)

weights_src = SUPPORT_DIR / 'weights'
if weights_src.is_dir():
    shutil.copytree(weights_src, REPO_DIR / 'weights')
else:
    (REPO_DIR / 'weights').mkdir()
    with zipfile.ZipFile(SUPPORT_DIR / 'weights.zip') as archive:
        archive.extractall(REPO_DIR / 'weights')
print('Materialized repo:', REPO_DIR)

In [ ]:
# Install only missing runtime dependencies from the attached offline wheels.
module_packages = {
    'tracksdata': 'tracksdata', 'zarr': 'zarr', 'pyscipopt': 'pyscipopt',
    'geff': 'geff', 'geff_spec': 'geff-spec', 'ilpy': 'ilpy',
    'polars': 'polars', 'blosc2': 'blosc2', 'dask': 'dask',
    'imagecodecs': 'imagecodecs', 'pyarrow': 'pyarrow',
    'rustworkx': 'rustworkx', 'sqlalchemy': 'sqlalchemy',
    'donfig': 'donfig', 'google_crc32c': 'google-crc32c',
    'bidict': 'bidict', 'psygnal': 'psygnal', 'rich': 'rich',
    'networkx': 'networkx', 'pydantic': 'pydantic',
    'pydantic_core': 'pydantic-core', 'annotated_types': 'annotated-types',
    'typing_extensions': 'typing-extensions',
    'typing_inspection': 'typing-inspection', 'markdown_it': 'markdown-it-py',
    'pygments': 'pygments', 'click': 'click', 'cloudpickle': 'cloudpickle',
    'fsspec': 'fsspec', 'partd': 'partd', 'locket': 'locket',
    'toolz': 'toolz', 'yaml': 'pyyaml', 'ndindex': 'ndindex',
    'msgpack': 'msgpack', 'numexpr': 'numexpr', 'deprecated': 'deprecated',
    'wrapt': 'wrapt', 'imageio': 'imageio', 'PIL': 'pillow',
    'tifffile': 'tifffile', 'lazy_loader': 'lazy-loader', 'tqdm': 'tqdm',
}
import importlib.util
missing = [package for module, package in module_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    wheels = SUPPORT_DIR / 'wheels'
    assert wheels.exists(), f'Missing packages {missing} and no offline wheels directory'
    # Never let pip replace Kaggle's NumPy/SciPy binary stack in a live kernel.
    cmd = [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--no-deps', '--find-links', str(wheels), *missing]
    subprocess.run(cmd, check=True)
# Verify the exact interpreter used by inference before spending GPU time.
probe = subprocess.run(
    [sys.executable, '-c', 'import numpy, scipy, scipy.spatial, tracksdata; print(numpy.__version__, scipy.__version__)'],
    text=True, capture_output=True,
)
assert probe.returncode == 0, 'Binary dependency check failed:\n' + probe.stderr
import numpy as np
import scipy
import scipy.spatial
print('Dependencies ready; NumPy/SciPy:', np.__version__, scipy.__version__)

In [ ]:
# Pick the shortest labeled video to minimize smoke-test runtime.
def video_shape(path):
    return tuple(json.loads((path / '0' / 'zarr.json').read_text())['shape'])

candidates = []
for zarr_path in sorted(TRAIN_DIR.glob('*.zarr')):
    if (TRAIN_DIR / f'{zarr_path.stem}.geff').exists():
        shape = video_shape(zarr_path)
        candidates.append((shape[0], int(shape[1] * shape[2] * shape[3]), zarr_path.stem, shape))
assert candidates, 'No paired .zarr/.geff training datasets found'
_, _, SAMPLE, SAMPLE_SHAPE = min(candidates)
print(f'Selected {SAMPLE}: shape={SAMPLE_SHAPE}; labeled candidates={len(candidates)}')

splits_path = REPO_DIR / 'local_smoke_split.json'
splits_path.write_text(json.dumps([{'split': 0, 'train': [], 'test': [SAMPLE]}], indent=2))
weights = REPO_DIR / 'weights/unet_transformer/split_0/edge_predictor_best.pth'
assert weights.exists(), weights

In [ ]:
# One-video UNet + transformer + ILP inference.
env = {**os.environ, 'PYTHONPATH': str(REPO_DIR / 'src'), 'USER': 'local_eval', 'BIOHUB_DATA_DIR': str(TRAIN_DIR)}
cmd = [
    sys.executable, 'scripts/predict_unet_transformer.py',
    '--data-dir', str(TRAIN_DIR),
    '--splits', str(splits_path),
    '--split', '0',
    '--weights', str(weights),
    '--unet-batch-size', str(UNET_BATCH_SIZE),
    '--det-threshold', str(DET_THRESHOLD),
    '--use-ilp',
]
print(' '.join(cmd))
started = time.time()
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
runtime_minutes = (time.time() - started) / 60
print(f'Inference finished in {runtime_minutes:.2f} min')

In [ ]:
# Official support-pack metric: adjusted edge Jaccard + 0.1 * division Jaccard.
sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'scripts'))
import tracksdata as td
from biohub_tracking.io import open_dataset
from biohub_tracking.metrics import evaluate, node_recall, per_sample_metrics, summarise
from geff import GeffMetadata

pred_path = REPO_DIR / f'predictions/local_eval/unet_transformer/split_0/{SAMPLE}.geff'
assert pred_path.exists(), pred_path
pred_loaded = td.graph.IndexedRXGraph.from_geff(pred_path)
pred_graph = pred_loaded[0] if isinstance(pred_loaded, tuple) else pred_loaded
ground_truth = open_dataset(TRAIN_DIR / SAMPLE, require_tracks=True, load_image=False)
result = evaluate(pred_graph, ground_truth.tracks, scale=ground_truth.scale, max_distance=MAX_DISTANCE_UM)
recall = node_recall(pred_graph, ground_truth.tracks) if pred_graph.num_nodes() and pred_graph.num_edges() else 0.0
gt_meta = GeffMetadata.read(TRAIN_DIR / f'{SAMPLE}.geff')
estimated_nodes = float((gt_meta.extra or {}).get('estimated_number_of_nodes', float('nan')))
row = {'dataset': SAMPLE, **per_sample_metrics(result, estimated_nodes, recall)}
summary = summarise([row])

print(json.dumps({
    'dataset': SAMPLE,
    'shape': SAMPLE_SHAPE,
    'runtime_minutes': round(runtime_minutes, 3),
    'score': summary['score'],
    'edge_jaccard': summary['edge_jaccard'],
    'adjusted_edge_jaccard': summary['adj_edge_jaccard'],
    'division_jaccard': summary['division_jaccard'],
    'node_recall': summary['node_recall'],
    'predicted_nodes': result.num_pred_nodes,
    'estimated_total_nodes': estimated_nodes,
    'edge_tp_fp_fn': [result.edge_tp, result.edge_fp, result.edge_fn],
    'division_tp_fp_fn': [result.division_tp, result.division_fp, result.division_fn],
}, indent=2))

result_path = Path('/kaggle/working/local_evaluator_result.json')
result_path.write_text(json.dumps({'sample': SAMPLE, 'shape': SAMPLE_SHAPE, 'runtime_minutes': runtime_minutes, 'row': row, 'summary': summary}, indent=2, default=float))
print('Saved:', result_path)

## Next validation step

Once this smoke test succeeds, the next version will use embryo-grouped holdouts and fold-specific checkpoints. Never use this single training-video score to select hyperparameters.